In [ ]:
import torch
import torch.nn as nn

class customDKT(nn.Module):

    def __init__(self, num_concepts, embed_dim=128, hidden_dim=256, dropout=0.2):
        super().__init__()
        self.num_concepts = num_concepts
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.interaction_embed = nn.Embedding(num_concepts * 2, embed_dim)
        self.lstm_layer = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.out_layer = nn.Linear(hidden_dim, num_concepts)

    def forward(self, c, r, is_repeat, train=True):
        B, L = c.shape
        x = c + self.num_concepts * r # (B, L)
        h, _ = self.lstm_layer(self.interaction_embed(x)) # h in (B, L, hidden_dim)
        h = self.dropout(h) # h in (B, L, hidden_dim)

        pos = torch.arange(L, device=c.device).unsqueeze(0).expand(B, L) # pos in (B, L)
        start = torch.where(is_repeat==0, pos, torch.zeros_like(pos)) # start in (B, L)
        start = start.cummax(dim=1).values # start in (B, L)

        h_shift = torch.zeros_like(h) # h_shift in (B, L, hidden_dim)
        h_shift[:, 1:, :] = h[:, :-1, :] # h_shift in (B, L, hidden_dim)

        batch_idx = torch.arange(B, device=c.device).unsqueeze(1) # batch_idx in (B, 1)
        h_prev = h_shift[batch_idx, start] # h_prev in (B, L, hidden_dim)

        if train:
            logits = self.out_layer(h_prev) # logits in (B, L, num_concepts)
        else:
            logits = self.out_layer(h_shift) # logits in (B, L, num_concepts)
        
        return logits


In [2]:
import torch
import torch.nn.functional as F
 
def repeat_flags(batch, split):
    """Cờ 'cùng câu hỏi với vị trí trước' cho DKT.py.
    - train/valid (train_valid_sequences.csv): dùng is_repeat gốc.
    - test (test_question_window_sequences.csv): KHÔNG dùng is_repeat, vì pyKT
      đã tách KC thứ 2+ sang dòng riêng; suy lại từ qidxs."""
    if split != "test":
        return batch["is_repeat"]
    q = batch["qidxs"]
    rep = torch.zeros_like(q)
    rep[:, 1:] = ((q[:, 1:] == q[:, :-1]) & (q[:, 1:] >= 0)).long()
    return rep
 
def group_start(rep):
    """Vị trí đầu của câu hỏi chứa t (cùng công thức với DKT.py)."""
    B, L = rep.shape
    pos = torch.arange(L, device=rep.device).expand(B, L)
    return torch.cummax(torch.where(rep == 0, pos, torch.zeros_like(pos)), dim=1).values


In [3]:
def step(model, batch, split):
    # Dataset phải trả thêm "is_repeat" (train/valid) và "qidxs" (test); giá trị
    # đệm -1 đổi về 0 trước khi tra embedding (vị trí đệm đã bị loại qua selectmasks).
    c = batch["concepts"].clamp(min=0)
    r = batch["responses"].clamp(min=0)
    sm = batch["selectmasks"]
    rep = repeat_flags(batch, split)
    logits = model(c, r, rep)                                   # (B, L, C)
    logit_t = logits.gather(2, c.unsqueeze(-1)).squeeze(-1)     # logit của KC tại t
    valid = (sm == 1) & (group_start(rep) > 0)      # bỏ câu hỏi đầu (chưa có lịch sử)
    loss = F.binary_cross_entropy_with_logits(logit_t[valid], r[valid].float())
    return loss, torch.sigmoid(logit_t), valid      # xác suất + mask, gom theo qidxs


In [4]:
@torch.no_grad()
def test_dkt_group_invariants(model, num_concepts=865, seed=0):
    torch.manual_seed(seed); model.eval()                       # eval: tắt dropout
    dev = next(model.parameters()).device
    rep = torch.tensor([[0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0]], device=dev)
    c = torch.randint(0, num_concepts, rep.shape, device=dev)
    r = torch.randint(0, 2, rep.shape, device=dev)
    base, st = model(c, r, rep), group_start(rep)
    # (1) mọi KC của cùng câu hỏi dùng chung hidden state -> logits giống nhau
    for t in range(rep.size(1)):
        assert torch.allclose(base[0, t], base[0, st[0, t]])
    # (2) đảo nhãn KC trước trong CÙNG câu hỏi không làm đổi dự đoán
    for t in (rep[0] == 1).nonzero().flatten().tolist():
        r2 = r.clone(); r2[0, t - 1] = 1 - r2[0, t - 1]
        assert torch.allclose(model(c, r2, rep)[0, t], base[0, t])
    # (3) đảo nhãn KC cuối của câu hỏi TRƯỚC phải làm đổi dự đoán (có dùng lịch sử)
    r3 = r.clone(); r3[0, 2] = 1 - r3[0, 2]     # vị trí 2: KC cuối câu hỏi 1
    assert not torch.allclose(model(c, r3, rep)[0, 3], base[0, 3])
    print("PASS: bất biến nhóm câu hỏi")
 
def test_flags_on_expanded_rows():
    # Dòng [KC 10, KC 21] lấy từ file test của pyKT (Bảng 2.1, dòng 1)
    b = {"is_repeat": torch.tensor([[0, 1]]), "qidxs": torch.tensor([[0, 1]])}
    assert group_start(repeat_flags(b, "test"))[0, 1] == 1   # đúng: dùng h[0]
    assert group_start(b["is_repeat"])[0, 1] == 0            # cách cũ: không có lịch sử
    print("PASS: cờ nhóm trên dòng test đã mở rộng")
 
@torch.no_grad()
def test_eval_uses_qidx_flags(model):
    # Test tích hợp: đi qua đúng hàm step() mà script đánh giá sử dụng
    model.eval(); dev = next(model.parameters()).device
    def T(x): return torch.tensor([x], device=dev)
    b = {"concepts": T([10, 21]), "responses": T([1, 0]), "selectmasks": T([-1, 1]),
         "is_repeat": T([0, 1]), "qidxs": T([0, 1])}        # Bảng 2.1, dòng 1
    _, p1, valid = step(model, b, "test")
    _, p2, _ = step(model, dict(b, responses=T([0, 0])), "test")  # đảo đáp án câu 1
    assert valid[0, 1] and not torch.allclose(p1[0, 1], p2[0, 1])
    print("PASS: bước test dùng lịch sử câu hỏi trước")


In [5]:
model = customDKT(865)

test_dkt_group_invariants(model)

RuntimeError: where expected condition to be a boolean tensor, but got a tensor with dtype Long